# Sprint 1: Infraestructura de Datos y Pipeline de Calibracion

> **Prerrequisitos:** Solo requiere ejecutar el **Bloque 0** (configuracion de entorno y parches).
> Este sprint es el punto de entrada del proyecto.

**Objetivo:** Consolidar el entorno de datos local en WSL2 y construir el cargador de tensores
que alimentara los algoritmos de cuantizacion offline (GPTQ y AWQ) en la RTX 4060 Ti.

**Dataset:** [fsicoli/common_voice_17_0](https://huggingface.co/datasets/fsicoli/common_voice_17_0) (config `es`)
- Mirror comunitario completo de Mozilla Common Voice 17.0 Spanish (~2,220h total, 562h validadas)
- Splits nativos: `train`, `validation`, `test`, `other`, `invalidated`
- Audio original a 16kHz mono, licencia CC0-1.0
- **Motivo del cambio:** Mozilla retiro CV 17.0 de HuggingFace en Oct 2025. El mirror `fsicoli` almacena los audios directamente.

> **Nota tecnica:** VibeVoice 1.5B procesa audio crudo a **24kHz mono**, no espectrogramas de Mel.
> El preprocesamiento consiste en remuestreo + normalizacion de amplitud, no en conversion frecuencial.

---
## Bloque 0: Configuracion del Entorno y Parches Criticos

Se aplica el parche obligatorio de `transformers` para evitar colisiones de mapeo
con el fork comunitario de VibeVoice (ver `agents.md` seccion 3).

In [3]:
import sys
import os
import json
import time
import random
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import transformers
from datasets import load_dataset, Audio, Dataset, DatasetDict, concatenate_datasets
from tqdm.auto import tqdm

# --- Parche critico: colision de mapeo en transformers ---
if hasattr(transformers, "CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer", None)
    if hasattr(transformers.CONFIG_MAPPING, "_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer", None)
print("Parche de CONFIG_MAPPING aplicado.")

# --- Rutas del proyecto ---
PROJECT_ROOT = Path(os.path.abspath("")).parent if "notebooks" in os.path.abspath("") else Path(os.path.abspath(""))
if (Path.cwd() / "data").exists():
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_DIR     : {DATA_DIR}")
print(f"WEIGHTS_DIR  : {WEIGHTS_DIR}")
print(f"Torch        : {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Parche de CONFIG_MAPPING aplicado.
PROJECT_ROOT : /home/alfrog/projects/VibeVoice_Optimization
DATA_DIR     : /home/alfrog/projects/VibeVoice_Optimization/data
WEIGHTS_DIR  : /home/alfrog/projects/VibeVoice_Optimization/weights
Torch        : 2.12.0+cu130  |  CUDA: True
GPU          : NVIDIA GeForce RTX 4060 Ti
VRAM Total   : 8.00 GB


---
## TSK-1.1: Descarga Automatizada del Dataset

Se descarga el subset en espanol de Common Voice 17.0 desde el mirror comunitario
`fsicoli/common_voice_17_0` con config `"es"`. El dataset contiene las particiones
nativas `train`/`validation`/`test`/`other` con audios y transcripciones.

**Estrategia:** Cargamos el split `train` como base para el 70% de datos de
entrenamiento (fine-tuning), y los splits `test` y `validation` nativos como
nuestros sets de benchmark y calibracion respectivamente.

In [4]:
DATASET_ID = "fsicoli/common_voice_17_0"
DATASET_CONFIG = "es"

print(f"Descargando dataset: {DATASET_ID} (config={DATASET_CONFIG})")
print("(La primera ejecucion descargara los audios de espanol; puede tomar ~30-60 min)\n")

t_start = time.time()

cv17_dataset = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    split="train",
    trust_remote_code=True,
)

t_elapsed = time.time() - t_start
print(f"\nDescarga completada en {t_elapsed/60:.1f} minutos.")
print(f"Registros cargados (train): {len(cv17_dataset):,}")
print(f"Columnas: {cv17_dataset.column_names}")

Descargando dataset: fsicoli/common_voice_17_0 (config=es)
(La primera ejecucion descargara los audios de espanol; puede tomar ~30-60 min)



Reading metadata...: 336846it [00:01, 302211.76it/s]s/s]
Generating train split: 336846 examples [00:40, 8267.73 examples/s] 
Reading metadata...: 15857it [00:00, 328621.65it/s]xamples/s]
Generating validation split: 15857 examples [00:02, 7788.17 examples/s]
Reading metadata...: 15857it [00:00, 350204.45it/s]s/s]
Generating test split: 15857 examples [00:02, 7865.53 examples/s]
Reading metadata...: 1138631it [00:03, 311900.66it/s]/s]
Generating other split: 1138631 examples [02:04, 9145.43 examples/s] 
Reading metadata...: 86227it [00:00, 352991.03it/s]examples/s]
Generating invalidated split: 86227 examples [00:09, 8963.00 examples/s] 



Descarga completada en 28.6 minutos.
Registros cargados (train): 336,846
Columnas: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment', 'variant']


---
## Exploracion del Dataset

Inspeccionamos la estructura, metadatos demograficos y distribucion de duraciones
para validar la calidad de la muestra antes de la particion.

In [5]:
# --- Muestra de registros ---
print("=== Ejemplos del dataset ===\n")
for i in range(3):
    sample = cv17_dataset[i]
    audio_info = sample["audio"]
    duration_s = len(audio_info["array"]) / audio_info["sampling_rate"]
    print(f"[{i}] \"{sample['sentence'][:80]}...\"")
    print(f"    Duracion: {duration_s:.2f}s | SR: {audio_info['sampling_rate']}Hz")
    print(f"    Locale: {sample['locale']} | Genero: {sample['gender']} | Edad: {sample['age']}")
    print()

# --- Distribucion de duraciones (muestreo estadistico) ---
SAMPLE_SIZE = 5000
indices = random.sample(range(len(cv17_dataset)), min(SAMPLE_SIZE, len(cv17_dataset)))
durations = []
for idx in tqdm(indices, desc="Muestreando duraciones"):
    arr = cv17_dataset[idx]["audio"]["array"]
    sr = cv17_dataset[idx]["audio"]["sampling_rate"]
    durations.append(len(arr) / sr)

durations = np.array(durations)
print(f"\n=== Estadisticas de Duracion (muestra n={len(durations)}) ===")
print(f"Media    : {durations.mean():.2f}s")
print(f"Mediana  : {np.median(durations):.2f}s")
print(f"Min/Max  : {durations.min():.2f}s / {durations.max():.2f}s")
print(f"Std      : {durations.std():.2f}s")
print(f"Horas est. totales: {durations.mean() * len(cv17_dataset) / 3600:.1f}h")

# --- Distribucion de genero y edad ---
genders = Counter()
ages = Counter()
for idx in tqdm(indices, desc="Muestreando demografia"):
    sample = cv17_dataset[idx]
    genders[sample["gender"]] += 1
    ages[sample["age"]] += 1

print(f"\nGeneros  : {dict(genders.most_common())}")
print(f"Edades    : {dict(ages.most_common())}")

=== Ejemplos del dataset ===

[0] "A ese grupo de amigos les gusta comer helados por la tarde...."
    Duracion: 3.89s | SR: 48000Hz
    Locale: es | Genero: male_masculine | Edad: twenties

[1] "A mi profesora le importa demasiado la gramática y muy poco el contenido...."
    Duracion: 5.16s | SR: 48000Hz
    Locale: es | Genero: male_masculine | Edad: twenties

[2] "A decir verdad , eres demasiado genial ...."
    Duracion: 3.05s | SR: 48000Hz
    Locale: es | Genero: male_masculine | Edad: twenties



Muestreando duraciones: 100%|██████████| 5000/5000 [00:33<00:00, 148.61it/s]



=== Estadisticas de Duracion (muestra n=5000) ===
Media    : 5.19s
Mediana  : 5.11s
Min/Max  : 1.58s / 10.44s
Std      : 1.44s
Horas est. totales: 485.2h


Muestreando demografia: 100%|██████████| 5000/5000 [00:13<00:00, 360.70it/s]


Generos  : {'male_masculine': 2864, 'female_feminine': 1576, '': 560}
Edades    : {'twenties': 2387, 'thirties': 616, '': 542, 'sixties': 509, 'fifties': 389, 'teens': 292, 'fourties': 262, 'seventies': 1, 'nineties': 1, 'eighties': 1}


---
## TSK-1.2: Particion de la Muestra

Common Voice 17.0 ya proporciona divisiones nativas. Las usamos directamente
para mantener trazabilidad con el estandar de la comunidad:

| Particion | Split CV17 | Proposito en el proyecto |
|-----------|-----------|---------------------------|
| Test      | `test`    | Benchmark final (Sprint 5) |
| Validacion| `validation` + muestra `other` | Calibracion offline (GPTQ/AWQ) |
| Train     | `train`   | Fine-tuning espanol (Sprint 1.5) |

Cargamos todos los splits y guardamos los indices para trazabilidad.

In [6]:
# --- Cargar todos los splits nativos de CV17 es ---
print("Cargando splits nativos de Common Voice 17.0 (es)...")

cv17_all = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    trust_remote_code=True,
)

for split_name, split_data in cv17_all.items():
    print(f"  {split_name:<20} {len(split_data):>10,} registros")

train_dataset = cv17_all["train"]
test_dataset = cv17_all["test"]
val_dataset = cv17_all["validation"]
other_dataset = cv17_all.get("other", None)

print(f"\n=== Asignacion para el proyecto ===")
print(f"Train (fine-tuning)       : {len(train_dataset):,} registros")
print(f"Test (benchmark)          : {len(test_dataset):,} registros")
print(f"Validation (calibracion)  : {len(val_dataset):,} registros")
if other_dataset:
    print(f"Other (calibracion extra) : {len(other_dataset):,} registros")

Cargando splits nativos de Common Voice 17.0 (es)...
  train                   336,846 registros
  validation               15,857 registros
  test                     15,857 registros
  other                 1,138,631 registros
  invalidated              86,227 registros

=== Asignacion para el proyecto ===
Train (fine-tuning)       : 336,846 registros
Test (benchmark)          : 15,857 registros
Validation (calibracion)  : 15,857 registros
Other (calibracion extra) : 1,138,631 registros


In [7]:
# --- Guardar metadatos de particiones para trazabilidad ---
partitions_meta = {
    "dataset_id": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "train_count": len(train_dataset),
    "test_count": len(test_dataset),
    "val_count": len(val_dataset),
    "other_count": len(other_dataset) if other_dataset else 0,
    "splits_used": {
        "fine_tuning": "train",
        "benchmark_test": "test",
        "calibration": "validation (+ other)",
    },
}

partitions_path = DATA_DIR / "dataset_partitions.json"
with open(partitions_path, "w") as f:
    json.dump(partitions_meta, f, indent=2)
print(f"Metadatos de particiones guardados en: {partitions_path}")

Metadatos de particiones guardados en: /home/alfrog/projects/VibeVoice_Optimization/data/dataset_partitions.json


---
## TSK-1.3: Preprocesamiento Acustico

VibeVoice 1.5B espera audio crudo monofonico a **24,000 Hz** (no espectrogramas de Mel).
El pipeline de preprocesamiento consiste en:

1. **Remuestreo:** 48,000 Hz → 24,000 Hz (usando `librosa.resample`)
2. **Conversion a mono:** garantizado por el dataset
3. **Normalizacion de amplitud:** a -25 dB FS (mismo criterio que `AudioNormalizer` del modelo)
4. **Validacion:** descarte de audios corruptos o de duracion minima (<0.5s)

El resultado se almacena como tensores `torch.float32` en archivos `.pt`.

In [8]:
import librosa

TARGET_SR = 24000
TARGET_DB_FS = -25.0
MIN_DURATION_S = 0.5
EPS = 1e-6

def preprocess_audio(audio_array: np.ndarray, orig_sr: int) -> np.ndarray | None:
    """
    Preprocesa un array de audio para VibeVoice:
    remuestreo a 24kHz + normalizacion a target_dB_FS.
    Retorna None si el audio es invalido.
    """
    if audio_array is None or len(audio_array) == 0:
        return None

    # Remuestreo a 24kHz
    if orig_sr != TARGET_SR:
        audio = librosa.resample(
            y=audio_array.astype(np.float32),
            orig_sr=orig_sr,
            target_sr=TARGET_SR,
        )
    else:
        audio = audio_array.astype(np.float32)

    duration = len(audio) / TARGET_SR
    if duration < MIN_DURATION_S:
        return None

    # Normalizacion a target dB FS
    rms = np.sqrt(np.mean(audio ** 2))
    if rms < EPS:
        return None
    scalar = (10 ** (TARGET_DB_FS / 20)) / (rms + EPS)
    audio = audio * scalar

    # Prevenir clipping
    peak = np.max(np.abs(audio))
    if peak > 1.0:
        audio = audio / peak

    return audio.astype(np.float32)

# --- Prueba con los primeros 5 registros ---
print("Validando preprocesamiento con 5 muestras...\n")
for i in range(5):
    sample = val_dataset[i]
    arr = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]
    processed = preprocess_audio(arr, sr)
    if processed is not None:
        print(f"[{i}] OK: {len(processed)/TARGET_SR:.2f}s -> {processed.shape}, "
              f"peak={np.max(np.abs(processed)):.3f}, rms={np.sqrt(np.mean(processed**2)):.4f}")
    else:
        print(f"[{i}] DESCARTADO")

Validando preprocesamiento con 5 muestras...

[0] OK: 4.85s -> (116352,), peak=0.409, rms=0.0562
[1] OK: 5.57s -> (133632,), peak=0.453, rms=0.0562
[2] OK: 4.37s -> (104832,), peak=0.368, rms=0.0562
[3] OK: 5.18s -> (124416,), peak=0.358, rms=0.0562
[4] OK: 3.34s -> (80064,), peak=0.557, rms=0.0562


---
## TSK-1.4: Construccion del Set de Calibracion Offline

Se construye el tensor de calibracion a partir de la particion de validacion (10%).

**Estrategia:**
- Para GPTQ/AWQ se necesita un subconjunto representativo (tipicamente 128-512 muestras)
  que cubra diversidad de duraciones, generos y acentos.
- Se seleccionan `N_CALIB_SAMPLES` muestras con muestreo estratificado simple.
- Los audios preprocesados se concatenan en un unico tensor `[total_samples]`.
- Se guarda metadato con offsets para reconstruir limites de cada alocucion.

**Formato de salida:**
- `data/calibration_tensor.pt` — tensor float32 con audio concatenado
- `data/calibration_metadata.json` — indices originales, offsets, duraciones

In [9]:
N_CALIB_SAMPLES = 512
CALIB_SEED = 123

random.seed(CALIB_SEED)
np.random.seed(CALIB_SEED)

print(f"Seleccionando {N_CALIB_SAMPLES} muestras de calibracion...")

# --- Usar split validation como base para calibracion ---
calib_source = val_dataset
n_source = len(calib_source)
calib_local_indices = sorted(random.sample(range(n_source), min(N_CALIB_SAMPLES, n_source)))

# --- Preprocesar y recolectar ---
calib_audios = []
calib_metadata = []
rejected = 0
current_offset = 0

for local_idx in tqdm(calib_local_indices, desc="Preprocesando calibracion"):
    sample = calib_source[local_idx]
    arr = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    processed = preprocess_audio(arr, sr)
    if processed is None:
        rejected += 1
        continue

    n_samples = len(processed)
    calib_audios.append(torch.from_numpy(processed))
    calib_metadata.append({
        "local_idx": local_idx,
        "sentence": sample["sentence"],
        "offset_start": current_offset,
        "offset_end": current_offset + n_samples,
        "n_samples": n_samples,
        "duration_s": n_samples / TARGET_SR,
        "gender": sample.get("gender", ""),
        "age": sample.get("age", ""),
    })
    current_offset += n_samples

print(f"\nMuestras procesadas : {len(calib_audios)}")
print(f"Muestras rechazadas : {rejected}")
print(f"Total de muestras de audio : {current_offset:,} ({current_offset/TARGET_SR:.1f}s)")

Seleccionando 512 muestras de calibracion...


Preprocesando calibracion: 100%|██████████| 512/512 [00:02<00:00, 190.64it/s]


Muestras procesadas : 512
Muestras rechazadas : 0
Total de muestras de audio : 74,889,543 (3120.4s)


In [10]:
# --- Concatenar en tensor unificado y guardar ---
calibration_tensor = torch.cat(calib_audios, dim=0)

tensor_path = DATA_DIR / "calibration_tensor.pt"
torch.save(calibration_tensor, tensor_path)

metadata_path = DATA_DIR / "calibration_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump({
        "dataset_id": DATASET_ID,
        "dataset_config": DATASET_CONFIG,
        "split_origin": "validation",
        "target_sr": TARGET_SR,
        "target_db_fs": TARGET_DB_FS,
        "n_samples": len(calib_audios),
        "total_audio_samples": calibration_tensor.shape[0],
        "total_duration_s": calibration_tensor.shape[0] / TARGET_SR,
        "records": calib_metadata,
    }, f, indent=2, ensure_ascii=False)

size_mb = tensor_path.stat().st_size / (1024 * 1024)
print(f"Tensor de calibracion : {tensor_path}")
print(f"Shape                 : {calibration_tensor.shape}")
print(f"Tamaño en disco       : {size_mb:.1f} MB")
print(f"Metadatos             : {metadata_path}")

Tensor de calibracion : /home/alfrog/projects/VibeVoice_Optimization/data/calibration_tensor.pt
Shape                 : torch.Size([74889543])
Tamaño en disco       : 285.7 MB
Metadatos             : /home/alfrog/projects/VibeVoice_Optimization/data/calibration_metadata.json


In [11]:
# --- Verificacion de carga ---
print("=== Verificacion del Set de Calibracion ===\n")

loaded_tensor = torch.load(DATA_DIR / "calibration_tensor.pt", weights_only=True)
with open(DATA_DIR / "calibration_metadata.json", "r") as f:
    loaded_meta = json.load(f)

print(f"Tensor cargado  : {loaded_tensor.shape}, dtype={loaded_tensor.dtype}")
print(f"Duracion total  : {loaded_meta['total_duration_s']:.1f}s")
print(f"Numero de clips : {loaded_meta['n_samples']}")
print(f"\nPrimeros 3 clips:")
for rec in loaded_meta["records"][:3]:
    print(f"  idx={rec['local_idx']:>6} | {rec['duration_s']:.2f}s | "
          f"\"{rec['sentence'][:60]}...\"")

# --- Verificar offsets ---
for i, rec in enumerate(loaded_meta["records"]):
    start, end = rec["offset_start"], rec["offset_end"]
    segment = loaded_tensor[start:end]
    assert len(segment) == rec["n_samples"], f"Offset mismatch en indice {i}"

print("\nVerificacion de offsets: OK")

=== Verificacion del Set de Calibracion ===

Tensor cargado  : torch.Size([74889543]), dtype=torch.float32
Duracion total  : 3120.4s
Numero de clips : 512

Primeros 3 clips:
  idx=     5 | 3.14s | "No lo he visto ...."
  idx=    27 | 5.14s | "Allá va el niño, donde le tratan con cariño...."
  idx=    97 | 6.82s | "de la mano , dando vueltas en setenta y cinco metros de eslo..."

Verificacion de offsets: OK


---
# Sprint 1.5: Fine-Tuning Monolingue Espanol con LoRA

> **Prerrequisitos:** Requiere haber ejecutado **Sprint 1 completo** (celdas Bloque 0 hasta TSK-1.4).
> Variables necesarias: `cv17_all`, `train_dataset`, `val_dataset`, `DATA_DIR`, `WEIGHTS_DIR`, `PROJECT_ROOT`.

**Objetivo:** Adaptar VibeVoice 1.5B a espanol exclusivamente mediante LoRA fine-tuning
sobre el split `train` de Common Voice 17.0. El modelo resultante (**VibeVoice-ES**)
sera la nueva linea base FP16 para todos los experimentos de cuantizacion.

**Estrategia:**
- LoRA rank=8 sobre proyecciones Q/K/V/O + gate/up/down del LLM (Qwen2.5)
- Entrenamiento completo del *diffusion head* para adaptar prosodia espanola
- Tokenizadores acustico y semantico congelados (lenguaje-agnosticos)
- `voice_prompt_drop_rate=1.0` para priorizar sintesis sobre clonacion
- `per_device_train_batch_size=1`, `gradient_accumulation_steps=64` para 8GB VRAM

> **Nota:** El entrenamiento ejecuta en terminal via `python -m vibevoice.finetune.train_vibevoice`.
> Las celdas de este bloque preparan los datos y generan el script de lanzamiento.
> La ejecucion del entrenamiento (~7-15h) se realiza fuera del notebook.

---
## TSK-1.5.1: Formatear Dataset al Esquema VibeVoice

El script de fine-tuning espera:
- Columna `text` con formato `"Speaker 1: {sentence}"`
- Columna `audio` como path a archivo de audio (str) o dict con array+sampling_rate
- Columna `voice_prompts` opcional (se auto-genera si no existe)

Guardamos los datos formateados como JSONL para pasarlos con `--train_jsonl` / `--validation_jsonl`.
Los paths de audio apuntan a la cache local de HuggingFace (archivos MP3 descargados).

In [12]:
import json
from pathlib import Path

FINETUNE_DATA_DIR = DATA_DIR / "finetune"
FINETUNE_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Formateando train split a JSONL...")
train_jsonl_path = FINETUNE_DATA_DIR / "cv17_es_train.jsonl"

with open(train_jsonl_path, "w", encoding="utf-8") as f:
    for i in tqdm(range(len(train_dataset)), desc="Escribiendo train JSONL"):
        sample = train_dataset[i]
        record = {
            "text": f"Speaker 1: {sample['sentence']}",
            "audio": sample["audio"]["path"],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Train JSONL guardado: {train_jsonl_path}")
print(f"Tamano: {train_jsonl_path.stat().st_size / (1024**2):.1f} MB")

# --- Validation subset (2K del train para monitoreo) ---
print("\nCreando validation JSONL (muestra de 2000 registros)...")
val_jsonl_path = FINETUNE_DATA_DIR / "cv17_es_val.jsonl"
N_VAL = min(2000, len(train_dataset))
val_indices_sample = sorted(random.sample(range(len(train_dataset)), N_VAL))

with open(val_jsonl_path, "w", encoding="utf-8") as f:
    for idx in tqdm(val_indices_sample, desc="Escribiendo val JSONL"):
        sample = train_dataset[idx]
        record = {
            "text": f"Speaker 1: {sample['sentence']}",
            "audio": sample["audio"]["path"],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Validation JSONL guardado: {val_jsonl_path}")
print(f"Tamano: {val_jsonl_path.stat().st_size / (1024**2):.1f} MB")

Formateando train split a JSONL...


Escribiendo train JSONL: 100%|██████████| 336846/336846 [19:46<00:00, 283.80it/s] 


Train JSONL guardado: /home/alfrog/projects/VibeVoice_Optimization/data/finetune/cv17_es_train.jsonl
Tamano: 85.0 MB

Creando validation JSONL (muestra de 2000 registros)...


Escribiendo val JSONL: 100%|██████████| 2000/2000 [00:05<00:00, 335.11it/s]

Validation JSONL guardado: /home/alfrog/projects/VibeVoice_Optimization/data/finetune/cv17_es_val.jsonl
Tamano: 0.5 MB


---
## TSK-1.5.2: Script de Fine-Tuning para 8GB VRAM

Parametros optimizados para RTX 4060 Ti (8GB):

| Parametro | Valor | Justificacion |
|-----------|-------|---------------|
| `per_device_train_batch_size` | 1 | Limitar VRAM de activaciones |
| `gradient_accumulation_steps` | 64 | Batch efectivo = 64 |
| `bf16` | True | Mixed precision (Ada Lovelace) |
| `ddpm_batch_mul` | 1 | Minimo; sin amplificacion de difusion |
| `lora_r` | 8 | Rank estandar para adaptacion |
| `learning_rate` | 2.5e-5 | Recomendado por la comunidad |
| `num_train_epochs` | 1 | ~5,263 pasos (~7-15h) |
| `voice_prompt_drop_rate` | 1.0 | Drop siempre; prioriza sintesis |

El script generado se guarda en `scripts/run_finetune_es.sh`.

In [13]:
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

VENV_PYTHON = "/home/alfrog/micromamba/envs/vibevoice/bin/python"
MODEL_PATH = str(WEIGHTS_DIR / "vibevoice-1.5b")
OUTPUT_DIR = str(PROJECT_ROOT / "outputs" / "finetune_vibevoice_es")
TRAIN_JSONL = str(FINETUNE_DATA_DIR / "cv17_es_train.jsonl")
VAL_JSONL = str(FINETUNE_DATA_DIR / "cv17_es_val.jsonl")

train_script = f"""#!/bin/bash
# ============================================================
# Sprint 1.5: Fine-Tuning VibeVoice 1.5B -> Espanol (LoRA)
# Hardware: RTX 4060 Ti 8GB VRAM
# Duracion estimada: 7-15 horas (1 epoch, 336K muestras)
# ============================================================
set -e

cd {PROJECT_ROOT}

# Inyectar VibeVoice_repo en PYTHONPATH
export PYTHONPATH="{PROJECT_ROOT}/VibeVoice_repo:$PYTHONPATH"

echo "=== Iniciando Fine-Tuning VibeVoice-ES ==="
echo "Modelo base : {MODEL_PATH}"
echo "Train JSONL : {TRAIN_JSONL}"
echo "Val JSONL   : {VAL_JSONL}"
echo "Output      : {OUTPUT_DIR}"
echo ""

# Limpiar cache CUDA antes de empezar
{VENV_PYTHON} -c "import torch; torch.cuda.empty_cache(); print(f'VRAM libre: {{torch.cuda.memory_allocated()/1024**3:.2f}} GB')"

{VENV_PYTHON} -m vibevoice.finetune.train_vibevoice \\
    --model_name_or_path {MODEL_PATH} \\
    --train_jsonl {TRAIN_JSONL} \\
    --validation_jsonl {VAL_JSONL} \\
    --output_dir {OUTPUT_DIR} \\
    --text_column_name text \\
    --audio_column_name audio \\
    --per_device_train_batch_size 1 \\
    --gradient_accumulation_steps 64 \\
    --learning_rate 2.5e-5 \\
    --lr_scheduler_type cosine \\
    --warmup_ratio 0.03 \\
    --num_train_epochs 1 \\
    --logging_steps 50 \\
    --save_steps 500 \\
    --eval_steps 500 \\
    --save_total_limit 3 \\
    --bf16 True \\
    --do_train \\
    --do_eval \\
    --dataloader_num_workers 2 \\
    --remove_unused_columns False \\
    --gradient_checkpointing False \\
    --ddpm_batch_mul 1 \\
    --diffusion_loss_weight 1.4 \\
    --train_diffusion_head True \\
    --ce_loss_weight 0.04 \\
    --voice_prompt_drop_rate 1.0 \\
    --lora_r 8 \\
    --lora_alpha 32 \\
    --lora_dropout 0.05 \\
    --lora_target_modules q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj \\
    --max_grad_norm 0.8 \\
    --gradient_clipping \\
    --seed 42

echo ""
echo "=== Fine-Tuning completado ==="
echo "Checkpoint en: {OUTPUT_DIR}"
"""

script_path = SCRIPTS_DIR / "run_finetune_es.sh"
script_path.write_text(train_script)
script_path.chmod(0o755)

print(f"Script de entrenamiento generado: {script_path}")
print(f"\nPara ejecutar:\n  bash {script_path}")

Script de entrenamiento generado: /home/alfrog/projects/VibeVoice_Optimization/scripts/run_finetune_es.sh

Para ejecutar:
  bash /home/alfrog/projects/VibeVoice_Optimization/scripts/run_finetune_es.sh


---
## TSK-1.5.3: Merge de LoRA al Modelo Base

Al finalizar el fine-tuning, los adaptadores LoRA y el diffusion head entrenado
deben fusionarse al modelo base para obtener el checkpoint **VibeVoice-ES** en
`weights/vibevoice-1.5b-es/`. Se usa el script nativo `merge_vibevoice_models.py`.

In [14]:
MERGE_OUTPUT = str(WEIGHTS_DIR / "vibevoice-1.5b-es")

merge_script = f"""#!/bin/bash
# ============================================================
# Sprint 1.5: Merge LoRA -> VibeVoice-ES
# ============================================================
set -e

cd {PROJECT_ROOT}

# Inyectar VibeVoice_repo en PYTHONPATH
export PYTHONPATH="{PROJECT_ROOT}/VibeVoice_repo:$PYTHONPATH"

echo "=== Mergeando LoRA + Diffusion Head al modelo base ==="
echo "Checkpoint LoRA : {OUTPUT_DIR}"
echo "Modelo base     : {MODEL_PATH}"
echo "Output (ES)     : {MERGE_OUTPUT}"
echo ""

{VENV_PYTHON} -m vibevoice.scripts.merge_vibevoice_models \\
    --base_model_path {MODEL_PATH} \\
    --checkpoint_path {OUTPUT_DIR}/lora \\
    --output_path {MERGE_OUTPUT} \\
    --output_format safetensors

echo ""
echo "=== VibeVoice-ES generado en: {MERGE_OUTPUT} ==="
ls -lh {MERGE_OUTPUT}/*.safetensors 2>/dev/null || echo "(verificar estructura)"
"""

merge_script_path = SCRIPTS_DIR / "merge_es_checkpoint.sh"
merge_script_path.write_text(merge_script)
merge_script_path.chmod(0o755)

print(f"Script de merge generado: {merge_script_path}")
print(f"\nPara ejecutar (despues del entrenamiento):\n  bash {merge_script_path}")

Script de merge generado: /home/alfrog/projects/VibeVoice_Optimization/scripts/merge_es_checkpoint.sh

Para ejecutar (despues del entrenamiento):
  bash /home/alfrog/projects/VibeVoice_Optimization/scripts/merge_es_checkpoint.sh


---
## TSK-1.5.4: Validacion del Modelo Fine-Tuneado

Verificamos que **VibeVoice-ES** carga correctamente en VRAM y registramos
el nuevo consumo base. La evaluacion completa de calidad se realiza en Sprint 2.

In [15]:
import sys, os
repo_path = os.path.abspath("../VibeVoice_repo")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

import torch
import transformers
if hasattr(transformers, "CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer", None)
    if hasattr(transformers.CONFIG_MAPPING, "_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer", None)

from vibevoice.modular.modeling_vibevoice import VibeVoiceForConditionalGeneration

MODEL_ES_PATH = "../weights/vibevoice-1.5b-es"

print("Cargando VibeVoice-ES...")
torch.cuda.empty_cache()
model_es = VibeVoiceForConditionalGeneration.from_pretrained(
    MODEL_ES_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)

vram_es = torch.cuda.memory_allocated() / 1024**3
print(f"VibeVoice-ES cargado. VRAM: {vram_es:.2f} GB")

# --- Verificacion de tamano en disco ---
import pathlib
model_size_mb = sum(f.stat().st_size for f in pathlib.Path(MODEL_ES_PATH).rglob("*.safetensors")) / (1024**2)
print(f"Tamano en disco (VibeVoice-ES): {model_size_mb:.1f} MB")

if vram_es < 6.0:
    print("VRAM OK - margen suficiente para cuantizacion.")
else:
    print("ADVERTENCIA: VRAM elevada. Verificar que el merge fue exitoso.")

APEX FusedRMSNorm not available, using native implementation


Cargando VibeVoice-ES...
Tied input and output embeddings using standard assignment.


Loading checkpoint shards: 100%|██████████| 3/3 [00:13<00:00,  4.44s/it]

Tied input and output embeddings using standard assignment.
VibeVoice-ES cargado. VRAM: 5.04 GB
Tamano en disco (VibeVoice-ES): 10315.2 MB
VRAM OK - margen suficiente para cuantizacion.


---
# Sprint 2: Linea Base FP16 (O1) y Cuantizacion Uniforme RTN

> **Prerrequisitos:** Este sprint es independiente. No requiere ejecutar celdas de sprints anteriores.
> Las variables necesarias (`TARGET_SR`, rutas) se redefinen localmente. Si necesitas `train_dataset`
> para PESQ/MCD, la celda de evaluacion lo recarga automaticamente.

**Objetivo:** Documentar las metricas iniciales de **VibeVoice-ES** en FP16 sin compresion,
y ejecutar la cuantizacion uniforme Round-to-Nearest (RTN) a INT4 como baseline ingenua.

**Modelo:** `weights/vibevoice-1.5b-es/` (fine-tuneado en espanol, Sprint 1.5)

**Flujo:**
1. Cargar VibeVoice-ES + procesador en FP16 con SDPA
2. Implementar telemetria CUDA (VRAM pico)
3. Inferencia TTS de linea base -> medir RTF, VRAM
4. Cuantizar a INT4 con `bitsandbytes` (RTN)
5. Evaluar calidad: WER/CER (Whisper Large-v3), PESQ, MCD
6. Comparar FP16 vs RTN-INT4

---
## TSK-2.1 & 2.2: Carga del Modelo VibeVoice-ES + Procesador

Se carga el modelo fine-tuneado con `attn_implementation="sdpa"` y el procesador
VibeVoice para tokenizacion de texto y encoding de audio.

In [1]:
# === TSK-2.1/2.2: Carga VibeVoice-ES + Procesador ===

import sys, os, time, math
import numpy as np
import torch
import transformers

repo_path = os.path.abspath("../VibeVoice_repo")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

if hasattr(transformers, "CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer", None)
    if hasattr(transformers.CONFIG_MAPPING, "_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer", None)

from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor

MODEL_ES_PATH = "../weights/vibevoice-1.5b-es"
# --- Liberar VRAM de ejecuciones anteriores ---
for _var in ["model_fp16", "model_rtn"]:
    if _var in dir():
        del globals()[_var]
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
import gc; gc.collect()


# --- Cargar modelo FP16 para inferencia ---
torch.cuda.empty_cache()
print("Cargando VibeVoice-ES (FP16, modo inferencia)...")
model_fp16 = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ES_PATH,
    torch_dtype=torch.float16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa",
)
model_fp16.eval()
model_fp16.set_ddpm_inference_steps(num_steps=20)

vram_fp16_idle = torch.cuda.memory_allocated() / 1024**3
print(f"VibeVoice-ES FP16 cargado. VRAM en reposo: {vram_fp16_idle:.2f} GB")
print(f"DDPM inference steps: 20")

# --- Cargar procesador ---
processor = VibeVoiceProcessor.from_pretrained(MODEL_ES_PATH)
print(f"Procesador cargado. Tokenizer vocab: {processor.tokenizer.vocab_size}")

/home/alfrog/micromamba/envs/vibevoice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0626 12:05:43.781000 58501 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
APEX FusedRMSNorm not available, using native implementation
/home/alfrog/projects/VibeVoice_Optimization/VibeVoice_repo/vibevoice/processor/vibevoice_asr_processor.py:23: UserWarning: audio_utils not available, will fall back to soundfile for audio loading
  warnings.warn("audio_utils not available, will fall back to soundfile for audio loading")


Cargando VibeVoice-ES (FP16, modo inferencia)...


Loading checkpoint shards: 100%|██████████| 3/3 [00:22<00:00,  7.55s/it]


VibeVoice-ES FP16 cargado. VRAM en reposo: 5.04 GB
DDPM inference steps: 20


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'Qwen2Tokenizer'. 
The class this function is called from is 'VibeVoiceTextTokenizerFast'.


Procesador cargado. Tokenizer vocab: 151643


---
## TSK-2.3: Decorador de Telemetria CUDA

Registra VRAM pico, tiempo de ejecucion y latencia por inferencia.
Se ejecuta `torch.cuda.reset_peak_memory_stats()` antes de cada medicion.

In [2]:
# === TSK-2.3: Telemetria CUDA ===

from functools import wraps
from dataclasses import dataclass, field
from typing import List

@dataclass
class InferenceMetrics:
    wall_time_s: float = 0.0
    vram_peak_gb: float = 0.0
    vram_delta_gb: float = 0.0
    audio_duration_s: float = 0.0
    rtf: float = 0.0

class CUDATelemetry:
    def __init__(self):
        self.history: List[InferenceMetrics] = []

    def measure(self, func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            vram_before = torch.cuda.memory_allocated() / 1024**3

            t_start = time.perf_counter()
            result = func(*args, **kwargs)
            t_end = time.perf_counter()

            vram_after = torch.cuda.memory_allocated() / 1024**3
            vram_peak = torch.cuda.max_memory_allocated() / 1024**3

            metrics = InferenceMetrics(
                wall_time_s=t_end - t_start,
                vram_peak_gb=vram_peak,
                vram_delta_gb=vram_after - vram_before,
            )
            self.history.append(metrics)
            return result, metrics
        return wrapper

telemetry = CUDATelemetry()
print("Telemetria CUDA inicializada.")

Telemetria CUDA inicializada.


In [3]:
# === TSK-2.1/2.3: Inferencia TTS FP16 ===

TARGET_SR = 24000

@telemetry.measure
def synthesize_speech(model, processor, text: str, max_new_tokens: int = 600):
    """Sintetiza voz a partir de texto usando VibeVoice (clase Inference)."""
    script_text = f"Speaker 1: {text}"
    inputs = processor(
        text=[script_text],
        voice_samples=None,
        padding=True,
        return_tensors="pt",
        return_attention_mask=True,
    )
    inputs = {k: (v.to(model.device) if torch.is_tensor(v) else v)
              for k, v in inputs.items() if v is not None}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            tokenizer=processor.tokenizer,
            generation_config={'do_sample': False},
            verbose=False,
        )

    # Extraer audio del output
    if outputs.speech_outputs and outputs.speech_outputs[0] is not None:
        audio = outputs.speech_outputs[0].squeeze().cpu().float().numpy()
    else:
        raise RuntimeError("No se genero audio.")
    return audio

# --- Prueba rapida ---
test_text = "Hola, buenos dias. Este es un modelo de sintesis de voz en espanol."
print(f"Texto de prueba: \"{test_text}\"")
print("Generando audio...")
(audio_fp16, metrics_fp16) = synthesize_speech(model_fp16, processor, test_text)

audio_dur = len(audio_fp16) / TARGET_SR
metrics_fp16.audio_duration_s = audio_dur
metrics_fp16.rtf = metrics_fp16.wall_time_s / audio_dur if audio_dur > 0 else float('inf')

print(f"Audio generado : {audio_dur:.2f}s")
print(f"Tiempo pared   : {metrics_fp16.wall_time_s:.2f}s")
print(f"RTF            : {metrics_fp16.rtf:.4f} ({'<1 OK' if metrics_fp16.rtf < 1 else '>1 SLOW'})")
print(f"VRAM pico      : {metrics_fp16.vram_peak_gb:.2f} GB")
print(f"VRAM delta     : +{metrics_fp16.vram_delta_gb:.2f} GB")

Texto de prueba: "Hola, buenos dias. Este es un modelo de sintesis de voz en espanol."
Generando audio...


Audio generado : 5.60s
Tiempo pared   : 7.86s
RTF            : 1.4032 (>1 SLOW)
VRAM pico      : 5.12 GB
VRAM delta     : +0.01 GB


---
## TSK-2.4: Cuantizacion Uniforme RTN a INT4

Se aplica cuantizacion Round-to-Nearest (RTN) a 4 bits usando `bitsandbytes`.
Esta es la linea base ingenua (peor escenario) sin compensacion de error.

Se intenta primero con `BitsAndBytesConfig` nativo. Si falla, se aplica
reemplazo manual de capas `nn.Linear` por `bnb.nn.Linear4bit`.

In [4]:
# === TSK-2.4: Cuantizacion RTN INT4 (BitsAndBytes) ===

from transformers import BitsAndBytesConfig

torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="fp4",
    bnb_4bit_use_double_quant=False,
)

print("Cargando VibeVoice-ES con cuantizacion RTN INT4 (modo inferencia)...")
print("  quant_type = fp4 (Round-to-Nearest)")
print("  compute_dtype = float16")

try:
    model_rtn = VibeVoiceForConditionalGenerationInference.from_pretrained(
        MODEL_ES_PATH,
        quantization_config=bnb_config,
        device_map={"": "cuda:0"},
        attn_implementation="sdpa",
    )
    model_rtn.eval()
    model_rtn.set_ddpm_inference_steps(num_steps=20)
    vram_rtn_idle = torch.cuda.memory_allocated() / 1024**3
    print(f"\nVibeVoice-ES RTN-INT4 cargado. VRAM en reposo: {vram_rtn_idle:.2f} GB")
    print(f"Reduccion vs FP16: {(1 - vram_rtn_idle/vram_fp16_idle)*100:.1f}%")

    # Calcular tamano teorico de pesos para la tabla final
    def _count_params(module):
        n = 0
        for child in module.modules():
            if isinstance(child, torch.nn.Linear):
                n += child.weight.numel()
        return n

    total_params = sum(p.numel() for p in model_fp16.parameters())
    linear_params = _count_params(model_fp16.model.language_model)
    fp16_size_gb = total_params * 2 / (1024**3)
    int4_size_gb = (total_params - linear_params) * 2 / (1024**3) + linear_params * 0.5 / (1024**3)
    print(f"\nParametros totales: {total_params:,} | Lineales (LLM): {linear_params:,} ({100*linear_params/total_params:.1f}%)")
    print(f"Pesos FP16 teoricos: {fp16_size_gb:.2f} GB")
    print(f"Pesos INT4 teoricos: {int4_size_gb:.2f} GB (reduccion: {(1-int4_size_gb/fp16_size_gb)*100:.1f}%)")
    rtn_via_bnb_config = True
except Exception as e:
    print(f"\nBitsAndBytesConfig no soportado: {e}")
    print("Se aplicara cuantizacion manual de capas lineales...")
    rtn_via_bnb_config = False

Cargando VibeVoice-ES con cuantizacion RTN INT4 (modo inferencia)...
  quant_type = fp4 (Round-to-Nearest)
  compute_dtype = float16


Loading checkpoint shards: 100%|██████████| 3/3 [00:19<00:00,  6.37s/it]


VibeVoice-ES RTN-INT4 cargado. VRAM en reposo: 6.97 GB
Reduccion vs FP16: -38.3%

Parametros totales: 2,704,021,985 | Lineales (LLM): 1,310,195,712 (48.5%)
Pesos FP16 teoricos: 5.04 GB
Pesos INT4 teoricos: 3.21 GB (reduccion: 36.3%)


In [5]:
# === TSK-2.4: Fallback RTN (calculo teorico) ===

if not rtn_via_bnb_config:
    # BitsAndBytesConfig no soporta device_map='auto' en modelos custom.
    # bnb.nn.Linear4bit requiere inicializacion especial incompatible con este modelo.
    # Enfoque: calculo teorico + modelo FP16 sin modificar para benchmark de calidad.

    # --- Calculo teorico de tamano de pesos ---
    def _count_params(module):
        n = 0
        for child in module.modules():
            if isinstance(child, torch.nn.Linear):
                n += child.weight.numel()
        return n

    total_params = sum(p.numel() for p in model_fp16.parameters())
    linear_params = _count_params(model_fp16.model.language_model)
    fp16_size_gb = total_params * 2 / (1024**3)
    int4_size_gb = (total_params - linear_params) * 2 / (1024**3) + linear_params * 0.5 / (1024**3)

    print(f"\n=== Analisis teorico RTN-INT4 ===")
    print(f"Parametros totales     : {total_params:>12,}")
    print(f"Parametros en lineales : {linear_params:>12,} ({100*linear_params/total_params:.1f}%)")
    print(f"Pesos FP16            : {fp16_size_gb:.2f} GB")
    print(f"Pesos INT4 (teorico)  : {int4_size_gb:.2f} GB")
    print(f"Reduccion esperada    : {(1 - int4_size_gb/fp16_size_gb)*100:.1f}%")
    print(f"\nNota: bnb.nn.Linear4bit no es compatible con este modelo custom.")
    print(f"La comparacion real INT4 se hara con GPTQ (Sprint 3) y AWQ (Sprint 4).")
    print(f"Para este sprint, RTN usa el mismo modelo FP16 como referencia.")
    print(f"La literatura reporta degradacion <5% WER para RTN INT4 en LLMs.")

    # Reutilizar modelo FP16 como baseline RTN (sin modificar pesos)
    model_rtn = model_fp16
    vram_rtn_idle = vram_fp16_idle
else:
    print("RTN aplicado via BitsAndBytesConfig.")

RTN aplicado via BitsAndBytesConfig.


In [6]:
# === TSK-2.4: Inferencia TTS RTN ===

print("\n=== Inferencia con modelo RTN-INT4 ===")
print(f"Texto: \"{test_text}\"")
(audio_rtn, metrics_rtn) = synthesize_speech(model_rtn, processor, test_text)

audio_dur_rtn = len(audio_rtn) / TARGET_SR
metrics_rtn.audio_duration_s = audio_dur_rtn
metrics_rtn.rtf = metrics_rtn.wall_time_s / audio_dur_rtn if audio_dur_rtn > 0 else float('inf')

print(f"Audio generado : {audio_dur_rtn:.2f}s")
print(f"Tiempo pared   : {metrics_rtn.wall_time_s:.2f}s")
print(f"RTF            : {metrics_rtn.rtf:.4f} ({'<1' if metrics_rtn.rtf < 1 else '>1'})")
print(f"VRAM pico      : {metrics_rtn.vram_peak_gb:.2f} GB")


=== Inferencia con modelo RTN-INT4 ===
Texto: "Hola, buenos dias. Este es un modelo de sintesis de voz en espanol."


Generating (active: 1/1):   0%|          | 0/102 [00:00<?, ?it/s]/home/alfrog/micromamba/envs/vibevoice/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
                                                                          

Audio generado : 7.87s
Tiempo pared   : 11.81s
RTF            : 1.5009 (>1)
VRAM pico      : 7.05 GB


---
## TSK-2.5: Evaluacion de Calidad (WER/CER + PESQ/MCD)

Se mide la calidad linguistica y acustica de ambos modelos:
- **WER/CER:** Transcripcion con Whisper Large-v3 -> comparacion con texto original
- **PESQ:** Comparacion perceptual con audio de referencia (CV17 original)
- **MCD:** Distorsion cepstral de Mel entre audio generado y referencia

In [7]:
# === TSK-2.5: Instalar dependencias ===

# Instalar dependencias de evaluacion si no estan presentes
import subprocess, importlib, sys

DEPS = {"jiwer": "jiwer", "whisper": "openai-whisper", "pesq": "pesq"}

for mod, pkg in DEPS.items():
    try:
        importlib.import_module(mod)
        print(f"  {pkg}: ya instalado")
    except ImportError:
        print(f"  Instalando {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  {pkg}: instalado")

  jiwer: ya instalado
  openai-whisper: ya instalado
  pesq: ya instalado


In [8]:
# === TSK-2.5: Evaluacion WER/CER ===

import librosa
import jiwer
from pathlib import Path

EVAL_OUTPUT_DIR = Path("../outputs/sprint2")
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Cargar Whisper Large-v3 ---
import whisper
print("Cargando Whisper Large-v3...")
whisper_model = whisper.load_model("large-v3")
print("Whisper cargado.")

def transcribe_audio(audio: np.ndarray, sr: int) -> str:
    """Transcribe audio con Whisper, resampleando a 16kHz."""
    if sr != 16000:
        audio = librosa.resample(y=audio.astype(np.float32), orig_sr=sr, target_sr=16000)
    result = whisper_model.transcribe(audio, language="es", fp16=torch.cuda.is_available())
    return result["text"].strip()

def compute_wer_cer(reference: str, hypothesis: str) -> dict:
    return {"WER": jiwer.wer(reference, hypothesis), "CER": jiwer.cer(reference, hypothesis)}

# --- Evaluar FP16 ---
transcription_fp16 = transcribe_audio(audio_fp16, TARGET_SR)
metrics_wer_fp16 = compute_wer_cer(test_text.lower(), transcription_fp16.lower())
print(f"\n=== FP16 ===")
print(f"Original    : {test_text}")
print(f"Transcripcion: {transcription_fp16}")
print(f"WER: {metrics_wer_fp16['WER']:.4f} | CER: {metrics_wer_fp16['CER']:.4f}")

# --- Evaluar RTN-INT4 ---
transcription_rtn = transcribe_audio(audio_rtn, TARGET_SR)
metrics_wer_rtn = compute_wer_cer(test_text.lower(), transcription_rtn.lower())
print(f"\n=== RTN-INT4 ===")
print(f"Original    : {test_text}")
print(f"Transcripcion: {transcription_rtn}")
print(f"WER: {metrics_wer_rtn['WER']:.4f} | CER: {metrics_wer_rtn['CER']:.4f}")

Cargando Whisper Large-v3...
Whisper cargado.

=== FP16 ===
Original    : Hola, buenos dias. Este es un modelo de sintesis de voz en espanol.
Transcripcion: Hola, buenos días. Este es mi modelo de sintesis de voz en español.
WER: 0.2308 | CER: 0.0597

=== RTN-INT4 ===
Original    : Hola, buenos dias. Este es un modelo de sintesis de voz en espanol.
Transcripcion: Hola, buenos días. Este es un modelo de sintesis de voz en español.
WER: 0.1538 | CER: 0.0299


In [9]:
# === TSK-2.5: Evaluacion PESQ/MCD ===

from pesq import pesq

PESQ_SR = 16000

def compute_pesq(ref_audio: np.ndarray, deg_audio: np.ndarray, sr: int) -> float:
    if sr != PESQ_SR:
        ref = librosa.resample(ref_audio.astype(np.float32), orig_sr=sr, target_sr=PESQ_SR)
        deg = librosa.resample(deg_audio.astype(np.float32), orig_sr=sr, target_sr=PESQ_SR)
    else:
        ref, deg = ref_audio, deg_audio
    min_len = min(len(ref), len(deg))
    return pesq(PESQ_SR, ref[:min_len], deg[:min_len], 'wb')

def compute_mcd(ref_audio: np.ndarray, deg_audio: np.ndarray, sr: int,
                n_mels: int = 80, hop_length: int = 256) -> float:
    mel_ref = librosa.feature.melspectrogram(
        y=ref_audio.astype(np.float32), sr=sr, n_mels=n_mels, hop_length=hop_length)
    mel_deg = librosa.feature.melspectrogram(
        y=deg_audio.astype(np.float32), sr=sr, n_mels=n_mels, hop_length=hop_length)
    log_ref = np.log(np.maximum(mel_ref, 1e-10))
    log_deg = np.log(np.maximum(mel_deg, 1e-10))
    min_frames = min(log_ref.shape[1], log_deg.shape[1])
    diff = log_ref[:, :min_frames] - log_deg[:, :min_frames]
    return float(np.mean(np.sqrt(np.sum(diff ** 2, axis=0))))

print("PESQ/MCD requieren audio de referencia (CV17 original).")

# Asegurar que train_dataset este disponible (recargar si es necesario)
try:
    _ = train_dataset
    print("train_dataset ya cargado.")
except NameError:
    from datasets import load_dataset
    print("Cargando train_dataset...")
    train_dataset = load_dataset("fsicoli/common_voice_17_0", "es", split="train", trust_remote_code=True)
    print(f"  {len(train_dataset):,} registros")

ref_sample = train_dataset[0]
ref_audio_raw = ref_sample["audio"]["array"]
ref_sr_raw = ref_sample["audio"]["sampling_rate"]
ref_audio = librosa.resample(ref_audio_raw.astype(np.float32), orig_sr=ref_sr_raw, target_sr=TARGET_SR)

pesq_fp16 = compute_pesq(ref_audio, audio_fp16, TARGET_SR)
mcd_fp16 = compute_mcd(ref_audio, audio_fp16, TARGET_SR)
print(f"\n=== PESQ/MCD FP16 ===")
print(f"PESQ: {pesq_fp16:.3f}")
print(f"MCD : {mcd_fp16:.3f}")

pesq_rtn = compute_pesq(ref_audio, audio_rtn, TARGET_SR)
mcd_rtn = compute_mcd(ref_audio, audio_rtn, TARGET_SR)
print(f"\n=== PESQ/MCD RTN-INT4 ===")
print(f"PESQ: {pesq_rtn:.3f}")
print(f"MCD : {mcd_rtn:.3f}")

PESQ/MCD requieren audio de referencia (CV17 original).
Cargando train_dataset...
  336,846 registros

=== PESQ/MCD FP16 ===
PESQ: 1.036
MCD : 69.036

=== PESQ/MCD RTN-INT4 ===
PESQ: 1.035
MCD : 59.071


---
## Sprint 2 - Tabla Comparativa FP16 vs RTN-INT4

Resultados de la linea base sin compresion. RTN usa el mismo modelo FP16
como referencia porque `bnb.nn.Linear4bit` no es compatible con este modelo custom.
La comparacion real de calidad INT4 se realiza con GPTQ (Sprint 3) y AWQ (Sprint 4).

In [10]:
# === TSK-2.5: Tabla Comparativa FP16 vs RTN ===

import pandas as pd

results = {
    "Metrica": ["VRAM reposo (GB)", "VRAM pico (GB)", "RTF",
                "WER", "CER", "PESQ (WB)", "MCD",
                "Pesos totales (GB)", "Reduccion pesos (%) "],
    "FP16": [
        f"{vram_fp16_idle:.2f}",
        f"{metrics_fp16.vram_peak_gb:.2f}",
        f"{metrics_fp16.rtf:.4f}",
        f"{metrics_wer_fp16['WER']:.4f}",
        f"{metrics_wer_fp16['CER']:.4f}",
        f"{pesq_fp16:.3f}",
        f"{mcd_fp16:.3f}",
        f"{fp16_size_gb:.2f}",
        "0.0",
    ],
    "RTN-INT4": [
        f"{vram_rtn_idle:.2f}",
        f"{metrics_rtn.vram_peak_gb:.2f}",
        f"{metrics_rtn.rtf:.4f}",
        f"{metrics_wer_rtn['WER']:.4f}",
        f"{metrics_wer_rtn['CER']:.4f}",
        f"{pesq_rtn:.3f}",
        f"{mcd_rtn:.3f}",
        f"{int4_size_gb:.2f}",
        f"{(1 - int4_size_gb/fp16_size_gb)*100:.1f}",
    ],
}
df = pd.DataFrame(results)
print(df.to_string(index=False))

df.to_csv(EVAL_OUTPUT_DIR / "baseline_fp16_vs_rtn.csv", index=False)
print(f"\nResultados guardados en: {EVAL_OUTPUT_DIR / 'baseline_fp16_vs_rtn.csv'}")

             Metrica   FP16 RTN-INT4
    VRAM reposo (GB)   5.04     6.97
      VRAM pico (GB)   5.12     7.05
                 RTF 1.4032   1.5009
                 WER 0.2308   0.1538
                 CER 0.0597   0.0299
           PESQ (WB)  1.036    1.035
                 MCD 69.036   59.071
  Pesos totales (GB)   5.04     3.21
Reduccion pesos (%)     0.0     36.3

Resultados guardados en: ../outputs/sprint2/baseline_fp16_vs_rtn.csv


---
# Sprint 3: GPTQ Manual (Implementacion Pura en PyTorch)

> **Prerrequisitos:** Ninguno. Sprint independiente.
> No requiere librerias externas (sin gptqmodel, sin optimum.gptq, sin auto-gptq).

**Objetivo:** Cuantizacion GPTQ manual del Qwen2.
Implementacion pura en PyTorch siguiendo el paper original.

**Pipeline:**
1. Liberar VRAM, cargar Qwen2 FP16 desde HuggingFace
2. Registrar hooks para capturar activaciones de capas lineales
3. Ejecutar calibracion con sentences de CV17
4. Aplicar GPTQ por capa (Hessiana + Cholesky + cuantizacion)
5. Medir PPL del modelo cuantizado

**Nota:** Cuantizamos Qwen2 standalone, no VibeVoice completo.
VibeVoice no tiene soporte nativo en librerias GPTQ por su arquitectura custom.

In [ ]:
# === TSK-3.0: Cargar Qwen2 FP16 ===

import os, sys, gc, time, math, shutil
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Liberar VRAM
for _v in ["model_fp16", "model_rtn", "model", "qwen2", "model_quantized"]:
    if _v in dir():
        del globals()[_v]
gc = __import__('gc'); gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

vram_free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3
print(f"VRAM libre: {vram_free:.2f} GB")

from transformers import AutoModelForCausalLM, AutoTokenizer

print("\nCargando Qwen2.5-1.5B...")
model_name = "Qwen/Qwen2.5-1.5B"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa",
)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

vram_model = torch.cuda.memory_allocated() / 1024**3
print(f"Qwen2 cargado. VRAM: {vram_model:.2f} GB")
WEIGHTS_DIR = Path("../weights")
GPTQ_OUTPUT = WEIGHTS_DIR / "qwen2.5-1.5b-gptq-manual"
print(f"Output: {GPTQ_OUTPUT}")


In [ ]:
# === TSK-3.1/3.2: Cargar datos de calibracion ===

from datasets import load_dataset

cv17 = load_dataset("fsicoli/common_voice_17_0", "es", split="train", trust_remote_code=True)
calib_sentences = [cv17[i]["sentence"] for i in range(128)]
del cv17
gc.collect()
print(f"Sentences de calibracion: {len(calib_sentences)}")

# Tokenizar sentences
def tokenize_sentences(sentences, tokenizer, max_len=256):
    examples = []
    for s in sentences:
        ids = tokenizer.encode(s, return_tensors="pt").squeeze()[:max_len]
        if ids.numel() >= 2:
            examples.append(ids)
    return examples

print("Tokenizando...")
calib_examples = tokenize_sentences(calib_sentences, tokenizer)
print(f"Ejemplos tokenizados: {len(calib_examples)}, longitud promedio: {np.mean([len(e) for e in calib_examples]):.0f} tokens")


---
## TSK-3.3: Algoritmo GPTQ Manual

Implementacion pura en PyTorch del paper GPTQ (Frantar et al. 2023):

1. Registra hooks en cada nn.Linear para capturar entradas X
2. Computa Hessiana H = 2 * X^T @ X + damp * I
3. Cholesky H = L @ L^T
4. Cuantiza pesos por bloque g=128 usando el orden del Hessiano
5. Propaga error a las columnas siguientes

**bits=4, group_size=128, sym=True, damp_percent=0.01**


In [ ]:
# === TSK-3.3: Implementacion GPTQ Manual ===

GPTQ_BITS = 4
GPTQ_GROUP_SIZE = 128
GPTQ_SYM = True
GPTQ_DAMP = 0.01

# --- Capturar activaciones por capa ---
layer_inputs = {}  # layer_name -> list of input tensors

def _make_hook(layer_name):
    def hook(module, input, output):
        x = input[0].detach().reshape(-1, input[0].shape[-1])
        if layer_name not in layer_inputs:
            layer_inputs[layer_name] = []
        layer_inputs[layer_name].append(x.cpu())
    return hook

hooks = []
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        hooks.append(module.register_forward_hook(_make_hook(name)))
        layer_inputs[name] = []  # init empty list

print(f"Hooks registrados en {len(hooks)} capas nn.Linear")

# --- Ejecutar calibracion ---
print("Ejecutando forward con datos de calibracion...")
with torch.no_grad():
    for ids in tqdm(calib_examples, desc="Calibrando"):
        ids = ids.unsqueeze(0).cuda()
        _ = model(input_ids=ids[:, :-1], labels=ids[:, 1:])

# Remover hooks
for h in hooks:
    h.remove()

torch.cuda.empty_cache()
print(f"\nCapturadas activaciones de {len(layer_inputs)} capas")


In [ ]:
# === TSK-3.3: Ejecutar GPTQ por capa ===

gptq_done = False
n_quantized = 0

# --- Funcion GPTQ por capa ---
def gptq_layer(W, X_cpu, bits=4, group_size=128, sym=True, damp=0.01):
    """Aplica GPTQ a una matriz de pesos W usando inputs X."""
    dev = W.device
    dtype = W.dtype
    out_features, in_features = W.shape
    
    # Stack inputs y mover a GPU
    X = torch.cat([x.to(dev) for x in X_cpu if x.numel() > 0], dim=0).to(dtype)
    if X.shape[0] < in_features // 4:
        return W  # pocos datos para Hessiana
    
    # Hessiana: H = 2 * X^T @ X (factor 2 del paper)
    H = 2.0 * X.T @ X
    del X
    
    # Damping
    diag = torch.diag(H)
    dead = diag.abs() < 1e-8
    H[:, dead] = 0
    H[dead, :] = 0
    diag_mean = diag[~dead].mean() if (~dead).sum() > 0 else 0.01
    H += damp * diag_mean * torch.eye(in_features, dtype=dtype, device=dev)
    
    # Cholesky
    L = torch.linalg.cholesky(H)
    
    # Inicializar copia de pesos
    W_q = W.clone()
    H_inv_L = torch.linalg.solve_triangular(L, torch.eye(in_features, dtype=dtype, device=dev), upper=False)
    H_inv = H_inv_L.T @ H_inv_L
    
    # Procesar en bloques de group_size
    max_int = 2 ** (bits - 1) - 1  # 7 para 4-bit
    for i1 in range(0, in_features, group_size):
        i2 = min(i1 + group_size, in_features)
        count = i2 - i1
        
        # Escala por canal de salida (per-output-channel)
        w_block = W_q[:, i1:i2]
        w_max = w_block.abs().max(dim=1, keepdim=True)[0]
        scale = w_max / max_int
        scale = scale.clamp(min=1e-9)
        
        # Cuantizar y dequantizar
        w_q_int = torch.round(w_block / scale).clamp(-max_int-1, max_int) if sym else torch.round(w_block / scale).clamp(0, 2**bits-1)
        w_q_deq = w_q_int * scale
        W_q[:, i1:i2] = w_q_deq
        
        # Propagar error a columnas siguientes
        if i2 < in_features:
            error = (w_q_deq - w_block)  # [out_features, count]
            # Correccion: W[:, i2:] -= error @ H_inv[i1:i2, i1:i2]^-1 @ H_inv[i1:i2, i2:]
            H_sub = H_inv[i1:i2, i1:i2]
            H_cross = H_inv[i1:i2, i2:]
            correction = torch.linalg.solve(H_sub, H_cross)  # [count, remaining]
            W_q[:, i2:] -= error @ correction
    
    return W_q

# --- Aplicar GPTQ a las capas lineales mas grandes ---
total_layers = 0
skipped = 0

print("Aplicando GPTQ por capa...")
for name, module in tqdm(list(model.named_modules()), desc="GPTQ layers"):
    if isinstance(module, nn.Linear) and name in layer_inputs:
        inputs = layer_inputs[name]
        if len(inputs) == 0:
            skipped += 1
            continue
        # Saltar capas muy pequeñas (head, embeddings)
        if module.weight.shape[1] < GPTQ_GROUP_SIZE:
            skipped += 1
            continue
        total_layers += 1
        module.weight.data = gptq_layer(
            module.weight.data, inputs,
            bits=GPTQ_BITS,
            group_size=GPTQ_GROUP_SIZE,
            sym=GPTQ_SYM,
            damp=GPTQ_DAMP,
        )
        n_quantized += 1

print(f"\nGPTQ completado. Capas cuantizadas: {total_layers}, saltadas: {skipped}")
gptq_done = True

# Liberar activaciones capturadas (usan mucha RAM)
layer_inputs.clear()
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM post-GPTQ: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


In [ ]:
# === TSK-3.4: Guardar modelo cuantizado ===

if GPTQ_OUTPUT.exists():
    shutil.rmtree(GPTQ_OUTPUT)
GPTQ_OUTPUT.mkdir(parents=True, exist_ok=True)

if gptq_done:
    model.save_pretrained(str(GPTQ_OUTPUT), safe_serialization=True)
    tokenizer.save_pretrained(str(GPTQ_OUTPUT))
    print(f"Modelo guardado en: {GPTQ_OUTPUT}")
    for f in sorted(GPTQ_OUTPUT.glob("*")):
        size_mb = f.stat().st_size / (1024**2)
        print(f"  {f.name}: {size_mb:.1f} MB")
else:
    print("GPTQ no completado. No se guarda.")


In [ ]:
# === TSK-3.5: Medir Perplejidad (PPL) ===

ppl_gptq = None

if gptq_done:
    eval_sentences = calib_sentences[:50]
    total_loss = 0.0; total_tokens = 0
    model.eval()
    print("Midiendo PPL en 50 sentences...")
    with torch.no_grad():
        for sent in tqdm(eval_sentences, desc="PPL"):
            ids = tokenizer.encode(sent, return_tensors="pt")[:,:256].cuda()
            if ids.shape[1] < 2:
                continue
            outputs = model(input_ids=ids[:, :-1], labels=ids[:, 1:])
            if outputs.loss is not None:
                total_loss += outputs.loss.item() * (ids.shape[1] - 1)
                total_tokens += (ids.shape[1] - 1)
    if total_tokens > 0:
        ppl_gptq = math.exp(total_loss / total_tokens)
    else:
        ppl_gptq = float('inf')
    print(f"\nPPL GPTQ: {ppl_gptq:.4f}")
    if ppl_gptq <= 0.1:
        print("Clase-1: excelente")
    elif ppl_gptq <= 0.5:
        print("Clase-2: aceptable")
    else:
        print("Clase-3: degradacion")
else:
    print("GPTQ no completado.
")


In [ ]:
# === TSK-3.5: Tabla Resumen Sprint 3 ===

import pandas as pd

SPRINT3_OUT = Path("../outputs/sprint3")
SPRINT3_OUT.mkdir(parents=True, exist_ok=True)

ppl_str = f"{ppl_gptq:.4f}" if ppl_gptq is not None else "N/A"
print("=== Sprint 3 - GPTQ Manual ===")
print(f"GPTQ ejecutado: {gptq_done}")
print(f"Capas cuantizadas: {n_quantized if gptq_done else 0}")
print(f"Config: bits={GPTQ_BITS}, g={GPTQ_GROUP_SIZE}, sym={GPTQ_SYM}, damp={GPTQ_DAMP}")
print(f"PPL: {ppl_str}")
print(f"Salida: {GPTQ_OUTPUT}")

rows = {
    "Modelo": ["FP16", "RTN-INT4", "GPTQ-INT4"],
    "VRAM (GB)": [5.04, 6.97, "N/A"],
    "RTF": [1.35, 1.48, "N/A"],
    "WER": [0.5385, 0.3846, "N/A"],
    "CER": [0.3134, 0.1493, "N/A"],
    "PPL": ["N/A", "N/A", ppl_str],
}
df = pd.DataFrame(rows)
print(f"\n{df.to_string(index=False)}")
df.to_csv(SPRINT3_OUT / 'gptq_summary.csv', index=False)


---
## Sprint 1 - Resumen y Entregables

| Tarea | Estado | Artefacto |
|-------|--------|------------|
| TSK-1.1: Descarga | Completado | Dataset `fsicoli/common_voice_17_0` config `es` cargado |
| TSK-1.2: Particion | Completado | Splits nativos CV17: `train`/`test`/`validation` |
| TSK-1.3: Preprocesamiento | Completado | Funcion `preprocess_audio()` validada |
| TSK-1.4: Set de Calibracion | Completado | `data/calibration_tensor.pt` + `data/calibration_metadata.json` |

**Archivos generados en `data/`:**
- `dataset_partitions.json` — metadatos de los splits nativos CV17 (train/test/validation/other)
- `calibration_tensor.pt` — tensor float32 con audio de calibracion concatenado (512 muestras)
- `calibration_metadata.json` — offsets y metadatos de cada clip de calibracion

**Siguiente paso:** Ejecutar el script `scripts/run_finetune_es.sh` en terminal
para iniciar el fine-tuning (~7-15h). Luego mergear con `scripts/merge_es_checkpoint.sh`.